# Notebook 6 - Sample Hygiene and Clean Controls
### Scientific Reports 35109da0

### What the diagnostic actually showed

An earlier hypothesis - that registries strip the payload before we download it - is **wrong**:

| | value |
|---|---|
| pairs with an empty diff | **1 of 997** |
| malicious updates adding a security signal | **52.7%** |
| `bioql 5.5.1 -> 5.5.5` | 2,210 credential accesses, 415 network calls, 45 taint chains |

The malicious code is present and readable. Three defects in **how the sample was built**
are what flatten the signal:

**1. Non-consecutive pairs.** `bioql 5.5.1 -> 5.5.5` skips four releases, so the diff measures
several releases of accumulated change rather than one update. That pair alone reports
321,384 added lines.

**2. A handful of packages dominate.** The top five packages supply 22% of all rows;
`claw-subagent-service` alone contributes 56.

**3. Every negative comes from a package that was itself compromised.** Controls are drawn
from the same packages as the positives, before or after the incident. A package like
`bioql` adds thousands of calls in every release, compromised or not - so
**44% of "benign" updates also add security signals**, against 52.7% of malicious ones.
An 8.7-point gap cannot support a classifier.

### What this notebook does

| stage | fix |
|---|---|
| 1 | drop pairs whose versions are not consecutive in registry history |
| 2 | drop diffs above a size ceiling (accumulated change, not an update) |
| 3 | cap how many rows any single package contributes |
| 4 | add negatives from packages **never** compromised, size- and popularity-matched |
| 5 | re-evaluate, with every filter reported as an ablation |

Stages 1-3 need no downloads and run in minutes. Stage 4 downloads archives (~60-90 min).

**Nothing here is tuned on the outcome.** Thresholds are set from the data distribution
before any model is fitted, and the unfiltered result is reported alongside.

## Setup

In [ ]:
!pip install esprima -q
import os, io, re, ast, sys, math, json, time, glob, shutil, zipfile, tarfile, warnings, difflib
from collections import Counter
warnings.filterwarnings("ignore")
import requests, numpy as np, pandas as pd

SEED = 42
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/paper_revision"
except Exception:
    WORK = os.path.abspath("./paper_revision")
os.makedirs(WORK, exist_ok=True)
P = lambda n: os.path.join(WORK, n)

def log(m):
    line = f"[{time.strftime('%H:%M:%S')}] {m}"
    print(line, flush=True)
    try: open(P("pipeline_log.txt"), "a").write(line + "\n")
    except Exception: pass

KEY = {"package":str,"ecosystem":str,"version_before":str,"version_after":str}
v5 = pd.read_csv(P("features_v5_diff.csv"), dtype=KEY)
v5["label"] = pd.to_numeric(v5.label, errors="coerce").fillna(0).astype(int)
log(f"features_v5_diff: {v5.shape}  packages={v5.package.nunique()}")
log(str(v5.groupby(["ecosystem","label"]).size()))

## Registry access

Needed for the consecutiveness check and for stage 4.

In [ ]:
NPM_URL  = "https://registry.npmjs.org/{p}"
PYPI_URL = "https://pypi.org/pypi/{p}/json"
HEADERS  = {"User-Agent": "supply-chain-research/1.0 (academic)"}
DELAY, RETRIES = 0.4, 3

def _req(url, want_json, timeout):
    for a in range(RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout); time.sleep(DELAY)
            if r.status_code == 200: return r.json() if want_json else r.content
            if r.status_code in (429,500,502,503): time.sleep(2**a); continue
            return None
        except requests.RequestException: time.sleep(2**a)
        except ValueError: return None
    return None

fetch_npm  = lambda p: _req(NPM_URL.format(p=p), True, 15)
fetch_pypi = lambda p: _req(PYPI_URL.format(p=p), True, 15)
dl         = lambda u: _req(u, False, 30) if u else None

def npm_versions(m):
    t = m.get("time", {}) or {}
    vs = [v for v in (m.get("versions", {}) or {}) if v not in ("created","modified") and v != "0.0.1-security"]
    pr = [(v, t.get(v)) for v in vs if t.get(v)]; pr.sort(key=lambda x: x[1]); return pr

def pypi_versions(m):
    pr = []
    for v, fs in (m.get("releases", {}) or {}).items():
        if not fs: continue
        ut = fs[0].get("upload_time_iso_8601") or fs[0].get("upload_time")
        if ut: pr.append((v, ut))
    pr.sort(key=lambda x: x[1]); return pr

_hist_cache = {}
def history(pkg, eco):
    k = (pkg, eco)
    if k in _hist_cache: return _hist_cache[k]
    m = fetch_npm(pkg) if eco == "npm" else fetch_pypi(pkg)
    h = (npm_versions(m) if eco == "npm" else pypi_versions(m)) if m else []
    _hist_cache[k] = h
    return h

def tarball_url(pkg, eco, v):
    if eco == "npm":
        m = fetch_npm(pkg)
        try: return m["versions"][v]["dist"]["tarball"]
        except (KeyError, TypeError): return None
    m = fetch_pypi(pkg)
    try:
        fs = m["releases"].get(v, [])
        if not fs: return None
        sd = [f for f in fs if f["url"].endswith((".tar.gz",".zip"))]
        return (sd[0] if sd else fs[0])["url"]
    except (KeyError, TypeError): return None

log("registry helpers ready")

## Stage 1 - consecutiveness

A version pair is only an *update* if the two releases are adjacent in the package's own
publication history. `bioql 5.5.1 -> 5.5.5` is four updates merged into one diff.

The check is cached per package, so it costs one API call per distinct package.

In [ ]:
CK_CONS = P("consecutive_flags.csv")

pairs = v5[["package","ecosystem","version_before","version_after"]].drop_duplicates()
prev = pd.read_csv(CK_CONS, dtype=KEY) if os.path.exists(CK_CONS) else pd.DataFrame()
done = {(str(a),str(b),str(c)) for a,b,c in
        zip(prev.package, prev.ecosystem, prev.version_after)} if len(prev) else set()
log(f"pairs to check: {len(pairs)} | already done: {len(done)}")

rows, st = [], Counter()
try:
    for n, r in enumerate(pairs.itertuples(index=False), 1):
        k = (str(r.package), str(r.ecosystem), str(r.version_after))
        if k in done: st["skipped"] += 1; continue
        h = history(r.package, r.ecosystem)
        vers = [v for v, _ in h]
        gap, consec = -1, 0
        if str(r.version_before) in vers and str(r.version_after) in vers:
            i, j = vers.index(str(r.version_before)), vers.index(str(r.version_after))
            gap = j - i
            consec = int(gap == 1)
            st["consecutive" if consec else "gap"] += 1
        else:
            st["version_missing"] += 1
        rows.append({"package": r.package, "ecosystem": r.ecosystem,
                     "version_before": r.version_before, "version_after": r.version_after,
                     "gap": gap, "consecutive": consec})
        if n % 50 == 0:
            pd.DataFrame(rows).to_csv(CK_CONS, mode="a",
                                      header=not os.path.exists(CK_CONS), index=False)
            rows = []
            log(f"  {n}/{len(pairs)} {dict(st)}")
except KeyboardInterrupt:
    log("interrupted - progress saved")
finally:
    if rows:
        pd.DataFrame(rows).to_csv(CK_CONS, mode="a",
                                  header=not os.path.exists(CK_CONS), index=False)

flags = pd.read_csv(CK_CONS, dtype=KEY).drop_duplicates(
    subset=["package","ecosystem","version_after"])
d = v5.merge(flags[["package","ecosystem","version_after","gap","consecutive"]],
             on=["package","ecosystem","version_after"], how="left")
d["consecutive"] = d.consecutive.fillna(0).astype(int)
log(f"consecutive pairs: {int(d.consecutive.sum())} of {len(d)} "
    f"({100*d.consecutive.mean():.1f}%)")
log(str(d.groupby("label").consecutive.mean().round(3)))

## Stage 2 - diff size ceiling

The ceiling is the 90th percentile of added lines, computed **before** any model is fitted
so it cannot be tuned to the answer. Anything above it is accumulated change, not one update.

In [ ]:
CEIL = float(d.added_lines.quantile(0.90))
log(f"added_lines: median={d.added_lines.median():.0f}  p90={CEIL:.0f}  max={d.added_lines.max():.0f}")
d["size_ok"] = (d.added_lines <= CEIL).astype(int)
log(f"within ceiling: {int(d.size_ok.sum())} of {len(d)}")
log("largest diffs being excluded:")
log(str(d.nlargest(5, "added_lines")[["package","version_before","version_after",
                                      "added_lines","label"]].to_string(index=False)))

## Stage 3 - per-package cap

The top five packages supply 22% of all rows. Rows are kept at most `CAP` per package,
sampled deterministically, so no single package can drive the result.

In [ ]:
CAP = 8
def cap_rows(df, cap=CAP, seed=SEED):
    out = []
    for (p, e), sub in df.groupby(["package","ecosystem"]):
        for lab, s2 in sub.groupby("label"):
            out.append(s2.sample(min(len(s2), cap), random_state=seed))
    return pd.concat(out).reset_index(drop=True)

clean = d[(d.consecutive == 1) & (d.size_ok == 1)].copy()
log(f"after consecutiveness + size filters: {len(clean)}")
clean = cap_rows(clean)
log(f"after per-package cap ({CAP}): {len(clean)}  packages={clean.package.nunique()}")
log(str(clean.groupby(["ecosystem","label"]).size()))
log(f"prevalence: {clean.label.mean():.3f}")

sig = [c for c in clean.columns if c.startswith("add_")]
log("fraction of rows adding ANY security signal (was 0.527 vs 0.440):")
log(f"  malicious: {(clean[clean.label==1][sig].sum(axis=1)>0).mean():.3f}")
log(f"  benign   : {(clean[clean.label==0][sig].sum(axis=1)>0).mean():.3f}")

## Stage 4 - controls from never-compromised packages

Every negative so far comes from a package that was itself compromised. A package that adds
thousands of calls in every release looks the same before and after the incident, which is
why 44% of benign updates also add security signals.

This stage draws consecutive update pairs from popular packages with **no** entry in the
malicious set - the contrast the classifier actually needs. They are matched to the positives
on ecosystem and archive size.

Set `RUN_STAGE4 = False` to skip and evaluate stages 1-3 only.

In [ ]:
RUN_STAGE4 = True
CK_CLEAN = P("clean_controls.csv")
N_WANT = 400          # target number of clean control pairs

SEED_PKGS = {
 "npm": ["lodash","chalk","commander","express","axios","react","debug","semver","yargs",
         "moment","uuid","rimraf","glob","minimist","tslib","dotenv","cors","body-parser",
         "mkdirp","classnames","prop-types","redux","webpack","eslint","jest","typescript",
         "node-fetch","ws","socket.io","ejs","handlebars","marked","nodemon","pm2"],
 "pypi": ["requests","flask","django","numpy","pandas","click","jinja2","pyyaml","six",
          "urllib3","certifi","idna","setuptools","pytest","boto3","sqlalchemy","pillow",
          "cryptography","attrs","packaging","tqdm","scipy","matplotlib","beautifulsoup4",
          "lxml","httpx","pydantic","rich","typer","loguru"],
}
compromised = set(zip(v5.package, v5.ecosystem))

In [ ]:
# feature extraction identical to notebook 5 so the new rows are comparable
SUSPICIOUS = {
  "net_http":        re.compile(r"\b(https?://|requests\.(get|post)|urllib|fetch\(|XMLHttpRequest|http\.request|axios)", re.I),
  "subprocess_exec": re.compile(r"\b(subprocess|child_process|os\.system|exec\(|eval\(|spawn\(|popen)", re.I),
  "env_cred_access": re.compile(r"(process\.env|os\.environ|getenv|\.aws/credentials|id_rsa|\.npmrc|\.pypirc|SSH_|API_KEY|SECRET)", re.I),
  "filesystem":      re.compile(r"\b(fs\.(write|read|unlink)|open\([^)]*['\"]w|shutil|os\.remove|writeFile)", re.I),
  "encoded_blob":    re.compile(r"(base64|atob|btoa|fromCharCode|eval\(.*decode|exec\(.*decode)", re.I),
  "install_hook":    re.compile(r"(preinstall|postinstall|setup\.py|cmdclass|__import__)", re.I),
  "obfuscation":     re.compile(r"(_0x[0-9a-f]{4,}|[A-Za-z0-9+/]{120,})", re.I),
}
SOURCES = re.compile(r"(process\.env|os\.environ|getenv|\.aws/credentials|id_rsa|"
                     r"\.npmrc|\.pypirc|SSH_|API_KEY|SECRET|TOKEN|PASSWORD|"
                     r"expanduser|homedir|hostname|platform\(\)|userInfo)", re.I)
SINKS   = re.compile(r"(https?://|fetch\(|axios|request\(|urlopen|requests\.(get|post)|"
                     r"\.send\(|socket|exec\(|spawn\(|system\(|popen|eval\()", re.I)
ENCODE  = re.compile(r"(base64|b64encode|atob|btoa|hex|fromCharCode|encodeURI|JSON\.stringify)", re.I)
LONG_STR = re.compile(r"['\"][A-Za-z0-9+/=_\-]{80,}['\"]")
HEXESC   = re.compile(r"(\\x[0-9a-fA-F]{2}|\\u[0-9a-fA-F]{4})")
MAX_LINES = 20000

def strip_root(n):
    p = n.split("/", 1); return p[1] if len(p) > 1 else n

def extract_map(arch, eco):
    if not arch: return {}
    exts = (".js",".ts",".json",".mjs",".cjs") if eco=="npm" else (".py",".cfg",".toml",".txt")
    out = {}
    try:
        try:
            with tarfile.open(fileobj=io.BytesIO(arch), mode="r:gz") as t:
                for m in t.getmembers():
                    if m.isfile() and m.name.endswith(exts) and m.size < 2_000_000:
                        f = t.extractfile(m)
                        if f: out[strip_root(m.name)] = f.read().decode("utf-8","ignore")
        except tarfile.ReadError:
            with zipfile.ZipFile(io.BytesIO(arch)) as z:
                for n in z.namelist():
                    if n.endswith(exts):
                        try: out[strip_root(n)] = z.read(n).decode("utf-8","ignore")
                        except Exception: pass
    except Exception: return {}
    return out

def added_lines_of(b, a):
    A = b.splitlines()[:MAX_LINES]; B = a.splitlines()[:MAX_LINES]
    if not A: return B
    out = []
    for tag,i1,i2,j1,j2 in difflib.SequenceMatcher(None, A, B, autojunk=False).get_opcodes():
        if tag in ("insert","replace"): out.extend(B[j1:j2])
    return out

def diff_of_pair(fb, fa):
    added, removed, touched, newf = [], [], 0, 0
    for path, at in fa.items():
        bt = fb.get(path)
        if bt is None:
            added.extend(at.splitlines()[:MAX_LINES]); newf += 1; touched += 1
        elif bt != at:
            added.extend(added_lines_of(bt, at)); removed.extend(added_lines_of(at, bt)); touched += 1
    return "\n".join(added), "\n".join(removed), touched, newf

def shannon(s):
    if not s: return 0.0
    n=len(s); return -sum((c/n)*math.log2(c/n) for c in Counter(s).values())

def split_blocks(text, eco):
    lines = text.splitlines(); blocks, cur, depth = [], [], 0
    if eco == "npm":
        for ln in lines:
            cur.append(ln); depth += ln.count("{") - ln.count("}")
            if depth <= 0 and len(cur) > 1: blocks.append("\n".join(cur)); cur=[]; depth=0
        if cur: blocks.append("\n".join(cur))
    else:
        for ln in lines:
            if ln and not ln[0].isspace() and cur: blocks.append("\n".join(cur)); cur=[]
            cur.append(ln)
        if cur: blocks.append("\n".join(cur))
    merged, buf = [], ""
    for b in blocks:
        buf = (buf+"\n"+b).strip() if buf else b
        if buf.count("\n") >= 3: merged.append(buf); buf=""
    if buf: merged.append(buf)
    return merged or ([text] if text else [])

def all_features(fb, fa, eco):
    added, removed, touched, newf = diff_of_pair(fb, fa)
    f = {}
    for k, rx in SUSPICIOUS.items():
        f[f"add_{k}"] = len(rx.findall(added)); f[f"rem_{k}"] = len(rx.findall(removed))
        f[f"net_{k}"] = f[f"add_{k}"] - f[f"rem_{k}"]
    n_add = max(added.count("\n")+1, 1)
    for k in SUSPICIOUS: f[f"addrate_{k}"] = round(f[f"add_{k}"]/n_add, 5)
    f["added_lines"] = added.count("\n")+1 if added else 0
    f["removed_lines"] = removed.count("\n")+1 if removed else 0
    f["files_touched"] = touched; f["files_new"] = newf; f["files_total"] = len(fa)
    f["churn_ratio"] = round(f["added_lines"]/max(f["removed_lines"],1),4)
    f["touched_frac"] = round(touched/max(len(fa),1),4)
    blocks = split_blocks(added, eco)
    n_src=n_snk=n_both=n_chain=n_enc=0
    for b in blocks:
        s=bool(SOURCES.search(b)); k=bool(SINKS.search(b)); n_src+=s; n_snk+=k
        if s and k:
            n_both+=1
            if ENCODE.search(b): n_enc+=1
            tainted={m_.group(1) for m_ in re.finditer(
                r"(?:const|let|var)?\s*([A-Za-z_$][\w$]{0,30})\s*=\s*([^;\n]{0,200})", b)
                if SOURCES.search(m_.group(2))}
            for sm in SINKS.finditer(b):
                seg=b[sm.start():min(len(b),sm.end()+300)]
                if any(re.search(r"\b"+re.escape(v)+r"\b", seg) for v in tainted):
                    n_chain+=1; break
    nb=max(len(blocks),1)
    f.update(tb_blocks=len(blocks), tb_src_blocks=n_src, tb_sink_blocks=n_snk,
             tb_both_blocks=n_both, tb_chain_blocks=n_chain, tb_encoded_flow=n_enc,
             tb_both_rate=round(n_both/nb,4), tb_chain_rate=round(n_chain/nb,4))
    lines=[l for l in added.splitlines() if l.strip()]; n=max(len(lines),1)
    idents=re.findall(r"[A-Za-z_$][\w$]{2,}", added); top=0
    for l in lines:
        if eco=="npm":
            if l and not l[0].isspace() and not l.lstrip().startswith(("//","/*","*","import","export")): top+=1
        else:
            if l and not l[0].isspace() and not l.lstrip().startswith(("#","import","from","def ","class ")): top+=1
    f.update(sc_added_entropy=round(shannon(added[:50000]),4),
             sc_long_strings=len(LONG_STR.findall(added)),
             sc_hex_escapes=len(HEXESC.findall(added)),
             sc_max_line_len=max((len(l) for l in lines), default=0),
             sc_mean_line_len=round(sum(len(l) for l in lines)/n,2),
             sc_toplevel_stmts=top, sc_toplevel_rate=round(top/n,4),
             sc_ident_entropy=round(shannon("".join(idents[:4000])),4),
             sc_mean_ident_len=round(sum(map(len,idents))/max(len(idents),1),2),
             sc_nonascii_rate=round(sum(ord(c)>127 for c in added[:50000])/max(min(len(added),50000),1),5))
    ih = {"ih_manifest_changed":0,"ih_scripts_added":0,"ih_install_hook_added":0,
          "ih_deps_added":0,"ih_setup_changed":0,"ih_hook_has_network":0,"ih_hook_has_exec":0}
    if eco=="npm":
        pb,pa = fb.get("package.json",""), fa.get("package.json","")
        if pb!=pa: ih["ih_manifest_changed"]=1
        try: jb=json.loads(pb) if pb else {}; ja=json.loads(pa) if pa else {}
        except Exception: jb,ja={},{}
        sb,sa=(jb.get("scripts") or {}),(ja.get("scripts") or {})
        ih["ih_scripts_added"]=max(len(sa)-len(sb),0)
        for hook in ("preinstall","install","postinstall","prepare"):
            if sa.get(hook) and sa.get(hook)!=sb.get(hook):
                ih["ih_install_hook_added"]=1; body=str(sa.get(hook))
                if SINKS.search(body): ih["ih_hook_has_network"]=1
                if re.search(r"(node |curl|wget|sh -c|bash|python)", body, re.I): ih["ih_hook_has_exec"]=1
        ih["ih_deps_added"]=len(set(ja.get("dependencies") or {})-set(jb.get("dependencies") or {}))
    else:
        sb="\n".join(v for k,v in fb.items() if k.endswith(("setup.py","setup.cfg","pyproject.toml")))
        sa="\n".join(v for k,v in fa.items() if k.endswith(("setup.py","setup.cfg","pyproject.toml")))
        if sb!=sa: ih["ih_setup_changed"]=ih["ih_manifest_changed"]=1
        add="\n".join(added_lines_of(sb,sa))
        if re.search(r"(cmdclass|install_requires|setup_requires|__import__|os\.system|subprocess)", add, re.I):
            ih["ih_install_hook_added"]=1
        if SINKS.search(add): ih["ih_hook_has_network"]=1
        if re.search(r"(exec\(|eval\(|os\.system|subprocess)", add, re.I): ih["ih_hook_has_exec"]=1
        ih["ih_deps_added"]=len(re.findall(r"install_requires", add))
    f.update(ih)
    return f

log("feature extraction ready (identical to notebook 5)")

In [ ]:
controls = pd.DataFrame()
if RUN_STAGE4:
    prev = pd.read_csv(CK_CLEAN, dtype=KEY) if os.path.exists(CK_CLEAN) else pd.DataFrame()
    done = {(str(a),str(b)) for a,b in zip(prev.package, prev.version_after)} if len(prev) else set()
    log(f"clean controls already built: {len(done)}")

    rows, st4, COLS = [], Counter(), None
    try:
        for eco, pkgs in SEED_PKGS.items():
            per_pkg = max(1, N_WANT // (len(SEED_PKGS) * max(len(pkgs), 1)) + 2)
            for pkg in pkgs:
                if (pkg, eco) in compromised: st4["is_compromised"] += 1; continue
                h = history(pkg, eco)
                if len(h) < 4: st4["too_few_releases"] += 1; continue
                vers = [v for v, _ in h]
                idxs = np.linspace(1, len(vers)-1, num=min(per_pkg, len(vers)-1), dtype=int)
                for i in sorted(set(idxs)):
                    vb, va = vers[i-1], vers[i]
                    if (str(pkg), str(va)) in done: st4["skipped"] += 1; continue
                    try:
                        fa = extract_map(dl(tarball_url(pkg, eco, va)), eco)
                        if not fa: st4["after_failed"] += 1; continue
                        fb = extract_map(dl(tarball_url(pkg, eco, vb)), eco)
                        if not fb: st4["before_failed"] += 1; continue
                        f = all_features(fb, fa, eco)
                        f.update(package=pkg, ecosystem=eco, label=0,
                                 version_before=vb, version_after=va,
                                 origin="clean_control", gap=1, consecutive=1)
                        if COLS is None: COLS = list(f.keys())
                        rows.append(f); st4["ok"] += 1
                    except Exception as e:
                        st4[f"err:{type(e).__name__}"] += 1
                    if len(rows) >= 20:
                        pd.DataFrame(rows).reindex(columns=COLS).to_csv(
                            CK_CLEAN, mode="a", header=not os.path.exists(CK_CLEAN), index=False)
                        rows = []
                        log(f"  {dict(st4)}")
    except KeyboardInterrupt:
        log("interrupted - progress saved")
    finally:
        if rows and COLS:
            pd.DataFrame(rows).reindex(columns=COLS).to_csv(
                CK_CLEAN, mode="a", header=not os.path.exists(CK_CLEAN), index=False)

    if os.path.exists(CK_CLEAN):
        controls = pd.read_csv(CK_CLEAN, dtype=KEY).drop_duplicates(
            subset=["package","ecosystem","version_after"])
        controls["label"] = 0
    log(f"clean controls: {len(controls)}  {dict(st4)}")

## Stage 5 - evaluation

Four datasets, same protocol throughout (package-grouped CV, class weighting, three seeds),
so the effect of each filter is visible rather than assumed.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, confusion_matrix, average_precision_score,
                             roc_auc_score)

FEAT = [c for c in v5.columns if c.startswith(("add_","rem_","net_","addrate_","tb_","sc_","ih_"))]
FEAT += [c for c in ["added_lines","removed_lines","files_touched","files_new",
                     "files_total","churn_ratio","touched_frac"] if c in v5.columns]
FEAT = [c for c in dict.fromkeys(FEAT)]

def evaluate(df, name, results):
    df = df.reset_index(drop=True)
    cols = [c for c in FEAT if c in df.columns]
    X = df[cols].fillna(0).values
    y, g = df.label.values, df.package.values
    if len(set(y)) < 2 or df.package.nunique() < 10:
        log(f"  {name:34s} skipped (n={len(df)})"); return None, None
    f1s, prs, rocs, fprs = [], [], [], []
    oof_last = None
    for s in (42, 7, 123):
        oof = np.zeros(len(X))
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, y, g):
            pos = y[tr].sum()
            w = np.where(y[tr]==1, (len(tr)-pos)/max(pos,1), 1.0)
            m = HistGradientBoostingClassifier(max_iter=300, random_state=s)
            m.fit(X[tr], y[tr], sample_weight=w)
            oof[te] = m.predict_proba(X[te])[:,1]
        pred = (oof>=0.5).astype(int)
        tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
        f1s.append(f1_score(y,pred,zero_division=0)); prs.append(average_precision_score(y,oof))
        rocs.append(roc_auc_score(y,oof)); fprs.append(fp/(fp+tn) if fp+tn else 0.0)
        oof_last = oof
    r = {"n":len(df), "packages":int(df.package.nunique()),
         "prevalence":round(float(y.mean()),4),
         "f1":round(float(np.mean(f1s)),4), "f1_sd":round(float(np.std(f1s)),4),
         "pr_auc":round(float(np.mean(prs)),4), "roc_auc":round(float(np.mean(rocs)),4),
         "fpr":round(float(np.mean(fprs)),4)}
    r["lift"] = round(r["pr_auc"] - r["prevalence"], 4)
    log(f"  {name:34s} n={r['n']:4d} prev={r['prevalence']:.3f}  F1={r['f1']:.3f}  "
        f"PR-AUC={r['pr_auc']:.3f}  ROC-AUC={r['roc_auc']:.3f}  lift={r['lift']:+.3f}")
    results[name] = r
    return r, oof_last

R6 = {}
log("--- filter ablation (lift = PR-AUC above chance) ---")
evaluate(d, "0. unfiltered (notebook 5)", R6)
evaluate(d[d.consecutive==1], "1. + consecutive only", R6)
evaluate(d[(d.consecutive==1)&(d.size_ok==1)], "2. + size ceiling", R6)
r_clean, oof = evaluate(clean, "3. + per-package cap", R6)

if len(controls):
    mixed = pd.concat([clean[clean.label==1], controls], ignore_index=True, sort=False)
    mixed["label"] = mixed.label.astype(int)
    log("--- with never-compromised controls ---")
    r_mix, oof_mix = evaluate(mixed, "4. malicious vs clean controls", R6)
    both = pd.concat([clean, controls], ignore_index=True, sort=False)
    both["label"] = both.label.astype(int)
    evaluate(both, "5. all negatives combined", R6)

In [ ]:
best_name = max((k for k in R6 if isinstance(R6[k], dict)),
                key=lambda k: R6[k].get("lift", -1))
best = R6[best_name]
log("="*78)
log(f"best configuration : {best_name}")
log(f"  PR-AUC {best['pr_auc']:.3f} against chance {best['prevalence']:.3f}  "
    f"-> lift {best['lift']:+.3f}")
log(f"  notebook 5 reference: lift +0.035")
log("="*78)
if best["lift"] > 0.15:
    log("VERDICT: sample hygiene was the problem. The signal is real and reportable.")
elif best["lift"] > 0.08:
    log("VERDICT: a genuine improvement, still modest for a headline detector claim.")
else:
    log("VERDICT: filtering does not recover the signal.")

R6["verdict"] = {"best": best_name, "lift": best.get("lift"),
                 "notebook5_lift": 0.035}
R6["filters"] = {"size_ceiling_p90": CEIL, "per_package_cap": CAP}
json.dump(R6, open(P("revision_results_v6.json"), "w"), indent=2, default=str)
clean.to_csv(P("features_v6_clean.csv"), index=False)
if len(controls): controls.to_csv(P("features_v6_controls.csv"), index=False)
log("saved revision_results_v6.json and features_v6_clean.csv")

---
## Reading the result

The number that matters is **lift** - PR-AUC above the class prevalence. Notebook 5 reached
**+0.035**, which is noise.

| lift | meaning |
|---|---|
| **> 0.15** | sample construction was the problem, not the features |
| 0.08 - 0.15 | genuine signal, modest for a headline claim |
| < 0.08 | filtering does not recover it |

Row 4 is the most informative: malicious updates against updates from packages that were
never compromised. If that lifts sharply while row 3 does not, the finding is that
**same-package controls are too hard a comparison** - itself a result worth reporting,
because it is the design three reviewers asked for.

Send back `revision_results_v6.json`.